#**ENSO Database: El Niño-Southern Oscillation Dynamical System**
###Created by Noah Riego

This notebook builds a relational database of the ENSO climate system, treating it as a chaotic dynamical system. It integrates 5 of the following data sources:

1. NOAA PSL - Sea surface temperature anomalies (Niño regions)
2. NCEP/NCAR - Atmospheric state variables (wind, pressure, radiation)
3. NOAA CPC - Oscillation indices (SOI, PDO, MJO)
4. USGS - Streamflow discharge (hydrological impacts)
5. NASA OISST - Gridded SST for spatial gradient computation

###SETUP

In [ ]:
# Install required packages
!pip install psycopg2-binary pandas requests b2sdk xarray netCDF4 --quiet

###IMPORTS

In [ ]:
# Install imports separately
import pandas as pd
import numpy as np
import requests
import io
import psycopg2
from psycopg2 import sql
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

###DATABASE CONNECTION CONFIGURATION

In [ ]:
# Aiven PostgreSQL credentials
DB_CONFIG = {
    'host': 'YOUR_AIVEN_HOST',
    'port': 12345,
    'database': 'defaultdb',
    'user': 'avnadmin',
    'password': 'YOUR_PASSWORD',
    'sslmode': 'require'
}

# Backblaze B2 credentials for intermediate file storage
B2_CONFIG = {
    'bucket_name': 'enso-data-raw',
    'app_key_id': '004c91daf1835b70000000002',
    'app_key': 'K004JP2oJRFoL3h7SQ4BcuxXvsWV3No'
}

###HELPER FUNCTIONS

In [ ]:
import time
import psycopg2

# Establish connection with retry logic
def get_db_connection(retries=5, delay=10):
    for attempt in range(retries):
        try:
            conn = psycopg2.connect(
                host=DB_CONFIG['host'],
                port=DB_CONFIG['port'],
                database=DB_CONFIG['database'],
                user=DB_CONFIG['user'],
                password=DB_CONFIG['password'],
                sslmode=DB_CONFIG['sslmode']
            )
            return conn
        except Exception as e:
            print(f"Connection attempt {attempt + 1} failed.")
            if attempt < retries - 1:
                print(f"Retrying in {delay} seconds...")
                time.sleep(delay)
            else:
                raise Exception("Database could not be reached after multiple attempts.") from e

# Test database connection
def test_db_connection():
    try:
        conn = get_db_connection()
        cursor = conn.cursor()
        cursor.execute("SELECT version();")
        version = cursor.fetchone()
        print("Database connection successful")
        print(f"PostgreSQL version: {version[0].split(',')[0]}")
        cursor.close()
        conn.close()
        return True
    except Exception as e:
        print(f"Database connection failed after retries: {e}")
        return False

# Test the connection
test_db_connection()

Database connection successful
PostgreSQL version: PostgreSQL 17.8 on x86_64-pc-linux-gnu


True

#EXTRACT PHASE

###SOURCE 1: NOAA PSL - Niño Region SST Indices

In [ ]:
print("\n[SOURCE 1] NOAA PSL - Niño Region SST Indices")
print("-" * 80)

# We need to extract 5 Niño region files:
# Niño 1+2 (far eastern equatorial Pacific), Niño 3 (central-eastern), Niño 3.4 (primary ENSO index), Niño 4 (central-western), and TNI (Trans-Niño Index)

# URL patterns for NOAA PSL data
NINO_URLS = {
    'nino12': 'https://psl.noaa.gov/data/correlation/nina1.data',
    'nino3': 'https://psl.noaa.gov/data/correlation/nina3.data',
    'nino34': 'https://psl.noaa.gov/data/correlation/nina34.data',
    'nino4': 'https://psl.noaa.gov/data/correlation/nina4.data',
    'tni': 'https://psl.noaa.gov/data/correlation/tni.data'
}

# We will extract NOAA PSL Niño region data from URL
# The data format is a fixed-width text file, and the first row contains year, then 12 monthly values
# Missing values are coded as -99.99 and the data spans roughly 1854-present. It returns DataFrame with year and 12 monthly columns

def extract_nino_region(url, region_name):
    print(f"  Fetching {region_name} data from NOAA PSL...")

    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()

        # Read the fixed-width format data
        # Skip header lines (first line is typically a title)
        lines = response.text.strip().split('\n')

        # Find where the actual data starts (skip header lines)
        data_start = 0
        for i, line in enumerate(lines):
            # Data lines start with a 4-digit year
            if line.strip() and line.strip()[0:4].isdigit():
                data_start = i
                break

        # Parse data lines
        data = []
        for line in lines[data_start:]:
            if not line.strip():
                continue

            # Split by whitespace
            values = line.split()

            if len(values) >= 13:  # year + 12 months
                try:
                    year = int(values[0])
                    monthly_values = [float(v) if v != '-99.99' else np.nan for v in values[1:13]]
                    data.append([year] + monthly_values)
                except ValueError:
                    continue

        # Create DataFrame
        columns = ['year'] + [f'month_{i}' for i in range(1, 13)]
        df = pd.DataFrame(data, columns=columns)

        print(f"  Successfully extracted {len(df)} years of {region_name} data ({df['year'].min()}-{df['year'].max()})")
        return df

    except Exception as e:
        print(f"  Error extracting {region_name}: {e}")
        return None

# Extract all Niño regions
nino_data = {}
for region, url in NINO_URLS.items():
    nino_data[region] = extract_nino_region(url, region.upper())

# Convert wide format to long format (one row per month)
def reshape_nino_to_long(df, region_name):
    if df is None:
        return None

    # Melt the dataframe
    df_long = df.melt(id_vars=['year'],
                      var_name='month',
                      value_name=region_name)

    # Extract month number from 'month_X' format
    df_long['month'] = df_long['month'].str.replace('month_', '').astype(int)

    # Create year_month key
    df_long['year_month'] = df_long['year'].astype(str) + '-' + df_long['month'].astype(str).str.zfill(2)

    # Drop rows with missing values
    df_long = df_long[df_long[region_name].notna()]

    # Keep only year_month and value columns
    df_long = df_long[['year_month', region_name]]

    return df_long

print("\n  Reshaping data to long format...")
nino_long = {}
for region, df in nino_data.items():
    nino_long[region] = reshape_nino_to_long(df, region)
    if nino_long[region] is not None:
        print(f"  {region.upper()}: {len(nino_long[region])} monthly observations")

# Merge all Niño regions into single DataFrame
print("\n  Merging all Niño regions...")
sst_observations = nino_long['nino12']
for region in ['nino3', 'nino34', 'nino4', 'tni']:
    sst_observations = sst_observations.merge(nino_long[region], on='year_month', how='outer')

# Filter to 1951-present (our target temporal coverage)
sst_observations['year'] = sst_observations['year_month'].str[:4].astype(int)
sst_observations = sst_observations[sst_observations['year'] >= 1951].copy()
sst_observations = sst_observations.drop('year', axis=1)

print(f"\n SOURCE 1 COMPLETE: {len(sst_observations)} monthly observations (1951-present)")
print(f"  Columns: {list(sst_observations.columns)}")
print(f"  Date range: {sst_observations['year_month'].min()} to {sst_observations['year_month'].max()}")
print(f"  Missing data: {sst_observations.isnull().sum().sum()} values")

# Preview the data
print("\n  Sample data:")
print(sst_observations.head())

print("\n Extract Phase for Source 1 is complete. Ready for Source 2.")


[SOURCE 1] NOAA PSL - Niño Region SST Indices
--------------------------------------------------------------------------------
  Fetching NINO12 data from NOAA PSL...
  Successfully extracted 79 years of NINO12 data (1948-2026)
  Fetching NINO3 data from NOAA PSL...
  Successfully extracted 79 years of NINO3 data (1948-2026)
  Fetching NINO34 data from NOAA PSL...
  Successfully extracted 79 years of NINO34 data (1948-2026)
  Fetching NINO4 data from NOAA PSL...
  Successfully extracted 79 years of NINO4 data (1948-2026)
  Fetching TNI data from NOAA PSL...
  Successfully extracted 78 years of TNI data (1948-2025)

  Reshaping data to long format...
  NINO12: 914 monthly observations
  NINO3: 914 monthly observations
  NINO34: 914 monthly observations
  NINO4: 914 monthly observations
  TNI: 936 monthly observations

  Merging all Niño regions...

 SOURCE 1 COMPLETE: 902 monthly observations (1951-present)
  Columns: ['year_month', 'nino12', 'nino3', 'nino34', 'nino4', 'tni']
  Date r

###SOURCE 2: NCEP/NCAR Reanalysis - Atmospheric State Variables

In [ ]:
print("\n[SOURCE 2] NCEP/NCAR Reanalysis - Atmospheric State Variables")
print("-" * 80)

# We need to fetch 5 atmospheric variable files from Backblaze B2
# These were manually downloaded from IRI Data Library and uploaded to B2 as tsv files
# The files are u850.tsv, u200.tsv, slp_darwin.tsv, slp_tahiti.tsv, and olr_monthly.tsv

from b2sdk.v2 import InMemoryAccountInfo, B2Api

def fetch_from_b2(bucket_name, file_name, app_key_id, app_key):
    print(f"  Fetching {file_name} from Backblaze B2...")

    try:
        # Initialize B2 API
        info = InMemoryAccountInfo()
        b2_api = B2Api(info)
        b2_api.authorize_account("production", app_key_id, app_key)

        # Get bucket
        bucket = b2_api.get_bucket_by_name(bucket_name)

        # Download file
        downloaded_file = bucket.download_file_by_name(file_name)
        file_buffer = io.BytesIO()
        bucket.download_file_by_name(file_name).save(file_buffer)
        file_buffer.seek(0)
        file_content = file_buffer.read().decode('utf-8')

        print(f"  Successfully fetched {file_name}")
        return file_content

    except Exception as e:
        print(f"  Error fetching {file_name}: {e}")
        return None

def parse_iri_tsv(file_content, variable_name):
    if file_content is None:
        return None

    lines = file_content.strip().split('\n')

    # Check if this is tab-separated format (multiple values on one line)
    if '\t' in lines[0]:
        # Format: all values on one line separated by tabs
        values = lines[0].split('\t')

        # Filter out empty strings and convert to float
        values = [float(v) for v in values if v.strip()]

        # Create time series starting from Jan 1949 (based on your u850 output)
        # IRI typically starts NCEP/NCAR reanalysis data at Jan 1949
        start_year = 1949
        start_month = 1

        data = []
        for i, value in enumerate(values):
            year = start_year + ((start_month - 1 + i) // 12)
            month = ((start_month - 1 + i) % 12) + 1
            year_month = f"{year}-{month:02d}"
            data.append([year_month, value])

        df = pd.DataFrame(data, columns=['year_month', variable_name])

    else:
        # Format: one value per line (no tabs, no time column)
        values = []
        for line in lines:
            line = line.strip()
            if line:
                try:
                    values.append(float(line))
                except ValueError:
                    continue

        # Create time series starting from Jan 1949
        start_year = 1949
        start_month = 1

        data = []
        for i, value in enumerate(values):
            year = start_year + ((start_month - 1 + i) // 12)
            month = ((start_month - 1 + i) % 12) + 1
            year_month = f"{year}-{month:02d}"
            data.append([year_month, value])

        df = pd.DataFrame(data, columns=['year_month', variable_name])

    return df

# Fetch and parse all atmospheric variables
atmospheric_files = {
    'u850': 'u850_monthly.tsv',
    'u200': 'u200_monthly.tsv',
    'slp_darwin': 'slp_darwin_monthly.tsv',
    'slp_tahiti': 'slp_tahiti_monthly.tsv',
    'olr': 'olr_monthly.tsv'
}

atmospheric_data = {}
for var_name, file_name in atmospheric_files.items():
    # Fetch from B2
    file_content = fetch_from_b2(
        B2_CONFIG['bucket_name'],
        file_name,
        B2_CONFIG['app_key_id'],
        B2_CONFIG['app_key']
    )

    # Parse the TSV
    if file_content:
        df = parse_iri_tsv(file_content, var_name)
        if df is not None:
            atmospheric_data[var_name] = df
            print(f"  Parsed {var_name}: {len(df)} monthly observations")
        else:
            print(f"  Failed to parse {var_name}")
    else:
        print(f"  Failed to fetch {var_name}")

# Merge all atmospheric variables into single DataFrame
print("\n  Merging atmospheric variables...")
atmospheric_state = atmospheric_data['u850']
for var in ['u200', 'slp_darwin', 'slp_tahiti', 'olr']:
    if var in atmospheric_data:
        atmospheric_state = atmospheric_state.merge(
            atmospheric_data[var],
            on='year_month',
            how='outer'
        )

# Compute Walker Circulation Index
# Walker Index = difference in zonal wind between upper and lower troposphere
# Positive = stronger Walker Circulation
print("\n  Computing derived Walker Circulation Index...")
atmospheric_state['walker_index'] = atmospheric_state['u200'] - atmospheric_state['u850']

# Filter to 1951-present to match Source 1
atmospheric_state['year'] = atmospheric_state['year_month'].str[:4].astype(int)
atmospheric_state = atmospheric_state[atmospheric_state['year'] >= 1951].copy()
atmospheric_state = atmospheric_state.drop('year', axis=1)

print(f"\n SOURCE 2 COMPLETE: {len(atmospheric_state)} monthly observations (1951-present)")
print(f"  Columns: {list(atmospheric_state.columns)}")
print(f"  Date range: {atmospheric_state['year_month'].min()} to {atmospheric_state['year_month'].max()}")
print(f"  Missing data: {atmospheric_state.isnull().sum().sum()} values")

# Preview the data
print("\n  Sample data:")
print(atmospheric_state.head())

print("\n Extract Phase for Source 2 is complete. Ready for Source 3.")


[SOURCE 2] NCEP/NCAR Reanalysis - Atmospheric State Variables
--------------------------------------------------------------------------------
  Fetching u850_monthly.tsv from Backblaze B2...
  Successfully fetched u850_monthly.tsv
  Parsed u850: 405 monthly observations
  Fetching u200_monthly.tsv from Backblaze B2...
  Successfully fetched u200_monthly.tsv
  Parsed u200: 405 monthly observations
  Fetching slp_darwin_monthly.tsv from Backblaze B2...
  Successfully fetched slp_darwin_monthly.tsv
  Parsed slp_darwin: 925 monthly observations
  Fetching slp_tahiti_monthly.tsv from Backblaze B2...
  Successfully fetched slp_tahiti_monthly.tsv
  Parsed slp_tahiti: 925 monthly observations
  Fetching olr_monthly.tsv from Backblaze B2...
  Successfully fetched olr_monthly.tsv
  Parsed olr: 64 monthly observations

  Merging atmospheric variables...

  Computing derived Walker Circulation Index...

 SOURCE 2 COMPLETE: 901 monthly observations (1951-present)
  Columns: ['year_month', 'u850',

###SOURCE 3: NOAA CPC - Oscillation Indices (SOI, PDO, MJO)

In [ ]:
print("\n[SOURCE 3] NOAA CPC - Oscillation Indices")
print("-" * 80)

# We will extract 3 index datasets:
# 1. SOI (Southern Oscillation Index) - monthly, 1951-present
# 2. PDO (Pacific Decadal Oscillation) - monthly, 1900-present
# 3. MJO (Madden-Julian Oscillation) RMM indices - daily, 1974-present

# SOI: Southern Oscillation Index
print("\n  Extracting SOI (Southern Oscillation Index)...")

SOI_URL = 'https://www.cpc.ncep.noaa.gov/data/indices/soi'

try:
    response = requests.get(SOI_URL, timeout=30)
    response.raise_for_status()

    lines = response.text.strip().split('\n')

    # SOI format is space-delimited, year followed by 12 monthly values
    # Skip header lines
    data = []
    for line in lines:
        if not line.strip():
            continue

        parts = line.split()

        # Data lines should have 13 values: year + 12 months
        if len(parts) >= 13:
            try:
                year = int(parts[0])
                monthly_values = [float(v) if v != '-999.9' else np.nan for v in parts[1:13]]

                # Create one row per month
                for month_idx, value in enumerate(monthly_values):
                    month = month_idx + 1
                    year_month = f"{year}-{month:02d}"
                    data.append([year_month, value])

            except ValueError:
                continue

    soi_df = pd.DataFrame(data, columns=['year_month', 'soi'])

    # Filter to 1951-present
    soi_df['year'] = soi_df['year_month'].str[:4].astype(int)
    soi_df = soi_df[soi_df['year'] >= 1951].copy()
    soi_df = soi_df.drop('year', axis=1)

    # Remove missing values
    soi_df = soi_df[soi_df['soi'].notna()]

    print(f"  SOI successfully extracted: {len(soi_df)} monthly observations ({soi_df['year_month'].min()} to {soi_df['year_month'].max()})")

except Exception as e:
    print(f"  Error extracting SOI: {e}")
    soi_df = pd.DataFrame(columns=['year_month', 'soi'])

# PDO: Pacific Decadal Oscillation
print("\n  Extracting PDO (Pacific Decadal Oscillation)...")

# PDO data from NOAA Earth System Research Lab
PDO_URL = 'https://www.ncei.noaa.gov/pub/data/cmb/ersst/v5/index/ersst.v5.pdo.dat'

try:
    response = requests.get(PDO_URL, timeout=30)
    response.raise_for_status()

    lines = response.text.strip().split('\n')

    # PDO format: year followed by 12 monthly values
    # Skip header lines (usually first 1-2 lines)
    data = []
    for line in lines:
        if not line.strip() or line.startswith('Year'):
            continue

        parts = line.split()

        if len(parts) >= 13:
            try:
                year = int(parts[0])
                monthly_values = [float(v) if v != '-99.99' and v != '99.99' else np.nan for v in parts[1:13]]

                # Create one row per month
                for month_idx, value in enumerate(monthly_values):
                    month = month_idx + 1
                    year_month = f"{year}-{month:02d}"
                    data.append([year_month, value])

            except ValueError:
                continue

    pdo_df = pd.DataFrame(data, columns=['year_month', 'pdo'])

    # Filter to 1951-present
    pdo_df['year'] = pdo_df['year_month'].str[:4].astype(int)
    pdo_df = pdo_df[pdo_df['year'] >= 1951].copy()
    pdo_df = pdo_df.drop('year', axis=1)

    # Remove missing values
    pdo_df = pdo_df[pdo_df['pdo'].notna()]

    print(f"  PDO successfully extracted: {len(pdo_df)} monthly observations ({pdo_df['year_month'].min()} to {pdo_df['year_month'].max()})")

except Exception as e:
    print(f"  Error extracting PDO: {e}")
    pdo_df = pd.DataFrame(columns=['year_month', 'pdo'])

# MJO: Madden-Julian Oscillation RMM Indices
print("\n  Extracting MJO (Madden-Julian Oscillation RMM indices)...")

# Fetch MJO data from Backblaze (BOM blocks automated requests)
try:
    mjo_content = fetch_from_b2(
        B2_CONFIG['bucket_name'],
        'mjo_rmm.txt',
        B2_CONFIG['app_key_id'],
        B2_CONFIG['app_key']
    )

    if mjo_content:
        lines = mjo_content.strip().split('\n')

        # MJO format: daily data with columns: year, month, day, RMM1, RMM2, phase, amplitude
        # Skip header lines
        data = []
        for line in lines[2:]:  # Skip first 2 header lines
            if not line.strip():
                continue

            parts = line.split()

            if len(parts) >= 7:
                try:
                    year = int(parts[0])
                    month = int(parts[1])
                    day = int(parts[2])
                    rmm1 = float(parts[3])
                    rmm2 = float(parts[4])
                    phase = int(parts[5]) if parts[5] != '0' else np.nan
                    amplitude = float(parts[6])

                    data.append([year, month, day, rmm1, rmm2, phase, amplitude])

                except ValueError:
                    continue

        mjo_daily = pd.DataFrame(data, columns=['year', 'month', 'day', 'mjo_rmm1', 'mjo_rmm2', 'mjo_phase', 'mjo_amplitude'])

        # Aggregate daily MJO data to monthly
        # - RMM1/RMM2: Arithmetic mean (these are continuous oscillating indices)
        # - Amplitude: Arithmetic mean (represents average MJO strength in the month)
        # - Phase: Mode (most common phase) — we use mode instead of mean because
        #   MJO phase is circular (phase 8 wraps to phase 1), so averaging
        #   would produce nonsensical results (e.g., avg of phase 1 and 8 = 4.5)

        mjo_daily['year_month'] = mjo_daily['year'].astype(str) + '-' + mjo_daily['month'].astype(str).str.zfill(2)

        mjo_monthly = mjo_daily.groupby('year_month').agg({
            'mjo_rmm1': 'mean',
            'mjo_rmm2': 'mean',
            'mjo_amplitude': 'mean',
            'mjo_phase': lambda x: x.mode()[0] if len(x.mode()) > 0 else np.nan  # Most common phase
        }).reset_index()

        # Filter to 1951-present
        mjo_monthly['year'] = mjo_monthly['year_month'].str[:4].astype(int)
        mjo_monthly = mjo_monthly[mjo_monthly['year'] >= 1951].copy()
        mjo_monthly = mjo_monthly.drop('year', axis=1)

        print(f"  MJO extracted and aggregated to monthly: {len(mjo_monthly)} observations ({mjo_monthly['year_month'].min()} to {mjo_monthly['year_month'].max()})")
    else:
        raise Exception("Failed to fetch MJO file from B2")

except Exception as e:
    print(f"  Error extracting MJO: {e}")
    mjo_monthly = pd.DataFrame(columns=['year_month', 'mjo_rmm1', 'mjo_rmm2', 'mjo_phase', 'mjo_amplitude'])

# Merge all oscillation indices
print("\n  Merging oscillation indices...")

oscillation_indices = soi_df.merge(pdo_df, on='year_month', how='outer')
oscillation_indices = oscillation_indices.merge(mjo_monthly, on='year_month', how='outer')

# Filter to 1951-present
oscillation_indices['year'] = oscillation_indices['year_month'].str[:4].astype(int)
oscillation_indices = oscillation_indices[oscillation_indices['year'] >= 1951].copy()
oscillation_indices = oscillation_indices.drop('year', axis=1)

# Remove duplicate year_month entries (keep first occurrence)
oscillation_indices = oscillation_indices.drop_duplicates(subset='year_month', keep='first')

print(f"\n SOURCE 3 COMPLETE: {len(oscillation_indices)} monthly observations (1951-present)")
print(f"  Columns: {list(oscillation_indices.columns)}")
print(f"  Date range: {oscillation_indices['year_month'].min()} to {oscillation_indices['year_month'].max()}")
print(f"  Missing data: {oscillation_indices.isnull().sum().sum()} values")

# Preview the data
print("\n  Sample data:")
print(oscillation_indices.head())

print("\n Extract Phase for Source 3 is complete. Ready for Source 4.")


[SOURCE 3] NOAA CPC - Oscillation Indices
--------------------------------------------------------------------------------

  Extracting SOI (Southern Oscillation Index)...
  SOI successfully extracted: 1800 monthly observations (1951-01 to 2025-12)

  Extracting PDO (Pacific Decadal Oscillation)...
  PDO successfully extracted: 902 monthly observations (1951-01 to 2026-02)

  Extracting MJO (Madden-Julian Oscillation RMM indices)...
  Fetching mjo_rmm.txt from Backblaze B2...
  Successfully fetched mjo_rmm.txt
  MJO extracted and aggregated to monthly: 597 observations (1974-06 to 2024-02)

  Merging oscillation indices...

 SOURCE 3 COMPLETE: 902 monthly observations (1951-present)
  Columns: ['year_month', 'soi', 'pdo', 'mjo_rmm1', 'mjo_rmm2', 'mjo_amplitude', 'mjo_phase']
  Date range: 1951-01 to 2026-02
  Missing data: 1222 values

  Sample data:
  year_month  soi   pdo  mjo_rmm1  mjo_rmm2  mjo_amplitude  mjo_phase
0    1951-01  2.5 -1.19       NaN       NaN            NaN       

###SOURCE 4: USGS - Streamflow Discharge (Hydrological Impacts)

In [ ]:
print("\n[SOURCE 4] USGS - Streamflow Discharge")
print("-" * 80)

# We'll extract monthly streamflow data from 3 ENSO-sensitive gauges:
# 1. Colorado River at Lees Ferry, AZ (gauge 09380000)
# 2. Columbia River at The Dalles, OR (gauge 14105700)
# 3. Sacramento River at Bend Bridge, CA (gauge 11377100)

# USGS provides a REST API for data retrieval

# Extraction parameters
#    - site_no: USGS site number
#    - site_name: Human-readable site name
#    - start_date: Start date in YYYY-MM-DD format
#    - end_date: End date in YYYY-MM-DD format
# Returns DataFrame with year_month and discharge columns

def extract_usgs_streamflow(site_no, site_name, start_date='1951-01-01', end_date='2026-02-01'):
    print(f"\n  Extracting {site_name} (Site {site_no})...")

    # USGS REST API URL for daily values
    # parameterCd=00060 is discharge in cubic feet per second
    url = f"https://waterservices.usgs.gov/nwis/dv/?format=rdb&sites={site_no}&parameterCd=00060&startDT={start_date}&endDT={end_date}&siteStatus=all"

    try:
        response = requests.get(url, timeout=60)
        response.raise_for_status()

        lines = response.text.strip().split('\n')

        # Find the header line (first non-comment line)
        header_idx = 0
        for i, line in enumerate(lines):
            if not line.startswith('#'):
                header_idx = i
                break

        # Column names are in the header line
        headers = lines[header_idx].split('\t')

        # Data starts 2 lines after header (skip format line)
        data_start = header_idx + 2

        daily_data = []
        for line in lines[data_start:]:
            if not line.strip() or line.startswith('#'):
                continue

            parts = line.split('\t')

            if len(parts) >= 3:
                try:
                    # Date is typically in column index 2 (format: YYYY-MM-DD)
                    date_str = parts[2]
                    year, month, day = date_str.split('-')

                    # Discharge value is typically in column index 3
                    discharge = float(parts[3])

                    daily_data.append([int(year), int(month), int(day), discharge])

                except (ValueError, IndexError):
                    continue

        df_daily = pd.DataFrame(daily_data, columns=['year', 'month', 'day', 'discharge'])

        # Aggregate to monthly means
        df_daily['year_month'] = df_daily['year'].astype(str) + '-' + df_daily['month'].astype(str).str.zfill(2)
        df_monthly = df_daily.groupby('year_month')['discharge'].mean().reset_index()
        df_monthly.columns = ['year_month', site_name]

        # Filter to 1951-present
        df_monthly['year'] = df_monthly['year_month'].str[:4].astype(int)
        df_monthly = df_monthly[df_monthly['year'] >= 1951].copy()
        df_monthly = df_monthly.drop('year', axis=1)

        print(f"  Extracted {len(df_monthly)} monthly observations ({df_monthly['year_month'].min()} to {df_monthly['year_month'].max()})")
        print(f"  Mean discharge: {df_monthly[site_name].mean():.2f} cfs")

        return df_monthly

    except Exception as e:
        print(f"  Error extracting {site_name}: {e}")
        return pd.DataFrame(columns=['year_month', site_name])

# Extract streamflow from all three gauges
gauges = {
    '09380000': 'colorado_discharge',
    '14105700': 'columbia_discharge',
    '11377100': 'sacramento_discharge'
}

streamflow_data = {}
for site_no, site_name in gauges.items():
    streamflow_data[site_name] = extract_usgs_streamflow(site_no, site_name)

# Merge all gauge data
print("\n  Merging streamflow data from all gauges...")
hydrological_impacts = streamflow_data['colorado_discharge']
for site in ['columbia_discharge', 'sacramento_discharge']:
    if site in streamflow_data and len(streamflow_data[site]) > 0:
        hydrological_impacts = hydrological_impacts.merge(
            streamflow_data[site],
            on='year_month',
            how='outer'
        )

# Compute discharge anomalies (standardized)
# Formula used: Anomaly = (value - mean) / std_dev
print("\n  Computing discharge anomalies (z-scores)...")
for col in ['colorado_discharge', 'columbia_discharge', 'sacramento_discharge']:
    if col in hydrological_impacts.columns:
        mean_val = hydrological_impacts[col].mean()
        std_val = hydrological_impacts[col].std()
        hydrological_impacts[f'{col}_anomaly'] = (hydrological_impacts[col] - mean_val) / std_val
        # Drop the raw discharge, keep only anomalies
        # Why drop raw values? Rivers have vastly different baseline flows
        # (Columbia ~181k cfs vs Sacramento ~12k cfs), making raw values
        # non-comparable. Z-scores enable cross-basin ENSO teleconnection analysis.
        hydrological_impacts = hydrological_impacts.drop(col, axis=1)

# Rename columns to remove '_anomaly' suffix
hydrological_impacts.columns = ['year_month'] + [col.replace('_anomaly', '') for col in hydrological_impacts.columns if col != 'year_month']

print(f"\n SOURCE 4 COMPLETE: {len(hydrological_impacts)} monthly observations (1951-present)")
print(f"  Columns: {list(hydrological_impacts.columns)}")
print(f"  Date range: {hydrological_impacts['year_month'].min()} to {hydrological_impacts['year_month'].max()}")
print(f"  Missing data: {hydrological_impacts.isnull().sum().sum()} values")

# Preview the data
print("\n  Sample data (discharge anomalies as z-scores):")
print(hydrological_impacts.head())

print("\n Extract Phase for Source 4 is complete. Ready for Source 5.")


[SOURCE 4] USGS - Streamflow Discharge
--------------------------------------------------------------------------------

  Extracting colorado_discharge (Site 09380000)...
  Extracted 902 monthly observations (1951-01 to 2026-02)
  Mean discharge: 13250.55 cfs

  Extracting columbia_discharge (Site 14105700)...
  Extracted 902 monthly observations (1951-01 to 2026-02)
  Mean discharge: 181158.90 cfs

  Extracting sacramento_discharge (Site 11377100)...
  Extracted 902 monthly observations (1951-01 to 2026-02)
  Mean discharge: 12234.01 cfs

  Merging streamflow data from all gauges...

  Computing discharge anomalies (z-scores)...

 SOURCE 4 COMPLETE: 902 monthly observations (1951-present)
  Columns: ['year_month', 'colorado_discharge', 'columbia_discharge', 'sacramento_discharge']
  Date range: 1951-01 to 2026-02
  Missing data: 0 values

  Sample data (discharge anomalies as z-scores):
  year_month  colorado_discharge  columbia_discharge  sacramento_discharge
0    1951-01          

###SOURCE 5: NASA OISST - Spatial Metrics from Gridded SST

WARNING: This will take around 10-15 minutes (sometimes even longer) to run because it's processing roughly 533 months of gridded SST data through OPeNDAP. Unfortunately this will require some patience.


In [ ]:
print("\n[SOURCE 5] NASA OISST - Spatial SST Gradient Metrics from NetCDF Grids")
print("-" * 80)

# We'll process NOAA OISST v2.1 gridded data via OPeNDAP
# OISST provides daily 0.25° resolution SST grids from Sep 1981 to present
# We'll compute monthly spatial metrics for the full period (approximately 533 months)

import xarray as xr
import calendar

print("\n  Accessing NOAA OISST v2.1 via NCEI THREDDS OPeNDAP server...")
print("  Computing spatial metrics for Sep 1981 - Jan 2026 (approximately 533 months)")
print("  This will take 10-15 minutes as we process gridded NetCDF data remotely...")

# First we must compute spatial SST metrics from OISST daily grids for a given month.
# This function uses OPeNDAP to access data remotely without full downloads.
# It returns a dictionary with zonal_gradient, meridional_asymmetry, cold_tongue_intensity

def compute_oisst_monthly_metrics(year, month):
    try:
        # Use mid-month (day 15) as representative for each month
        day = 15
        date_str = f"{year}{month:02d}{day:02d}"
        year_month_dir = f"{year}{month:02d}"

        # Correct NCEI THREDDS OPeNDAP URL (without .html)
        url = f"https://www.ncei.noaa.gov/thredds/dodsC/OisstBase/NetCDF/V2.1/AVHRR/{year_month_dir}/oisst-avhrr-v02r01.{date_str}.nc"

        # Open dataset with xarray
        ds = xr.open_dataset(url)

        # Extract SST variable (squeeze out time and zlev dimensions)
        sst = ds['sst'].squeeze()

        # OISST uses 0-360 longitude convention
        # Equatorial Pacific: 5S-5N, 120E-280E (280E = 80W)

        # Compute spatial metrics:

        # 1. Zonal gradient: Western Pacific (140E-160E) - Eastern Pacific (260E-280E = 100W-80W)
        sst_west = sst.sel(lat=slice(-5, 5), lon=slice(140, 160)).mean()
        sst_east = sst.sel(lat=slice(-5, 5), lon=slice(260, 280)).mean()
        zonal_gradient = float(sst_west.values - sst_east.values)

        # 2. Meridional asymmetry: Northern (2N-5N) - Southern (5S-2S)
        #    Over central-eastern Pacific (160E-240E = 160E-120W)
        sst_north = sst.sel(lat=slice(2, 5), lon=slice(160, 240)).mean()
        sst_south = sst.sel(lat=slice(-5, -2), lon=slice(160, 240)).mean()
        meridional_asymmetry = float(sst_north.values - sst_south.values)

        # 3. Cold tongue intensity: Minimum SST in equatorial band (5S-5N, 120E-280E)
        sst_equatorial = sst.sel(lat=slice(-5, 5), lon=slice(120, 280))
        cold_tongue_intensity = float(sst_equatorial.min().values)

        ds.close()

        return {
            'zonal_gradient': zonal_gradient,
            'meridional_asymmetry': meridional_asymmetry,
            'cold_tongue_intensity': cold_tongue_intensity,
            'quality_flag': 1
        }

    except Exception as e:
        # Return NaN if data access fails
        return {
            'zonal_gradient': np.nan,
            'meridional_asymmetry': np.nan,
            'cold_tongue_intensity': np.nan,
            'quality_flag': 0
        }

# Process all months from Sep 1981 to Jan 2026
spatial_data = []

start_year, start_month = 1981, 9
end_year, end_month = 2026, 1

current_year = start_year
current_month = start_month

months_processed = 0
successful = 0

print("\n  Processing spatial metrics month by month...")

while (current_year < end_year) or (current_year == end_year and current_month <= end_month):
    year_month = f"{current_year}-{current_month:02d}"

    # Compute metrics
    metrics = compute_oisst_monthly_metrics(current_year, current_month)

    spatial_data.append({
        'year_month': year_month,
        **metrics
    })

    if metrics['quality_flag'] == 1:
        successful += 1

    months_processed += 1
    if months_processed % 50 == 0:
        print(f"    Processed {months_processed} months ({successful} successful)...")

    # Increment month
    current_month += 1
    if current_month > 12:
        current_month = 1
        current_year += 1

spatial_metrics = pd.DataFrame(spatial_data)

print(f"\n SOURCE 5 COMPLETE: {len(spatial_metrics)} monthly observations (1981-2026)")
print(f"  Columns: {list(spatial_metrics.columns)}")
print(f"  Date range: {spatial_metrics['year_month'].min()} to {spatial_metrics['year_month'].max()}")
print(f"  Missing data: {spatial_metrics[['zonal_gradient', 'meridional_asymmetry', 'cold_tongue_intensity']].isnull().sum().sum()} values")
print(f"  Quality flags: {(spatial_metrics['quality_flag'] == 1).sum()} good, {(spatial_metrics['quality_flag'] == 0).sum()} missing")

# Preview the data
print("\n  Sample data:")
print(spatial_metrics.head(10))

print("EXTRACT PHASE IS NOW FULLY COMPLETE. All 5 sources successfully extracted!")
print("  Source 1: NOAA PSL ERSSTv5 (SST observations)")
print("  Source 2: NCEP/NCAR Reanalysis (Atmospheric state)")
print("  Source 3: NOAA CPC (Oscillation indices)")
print("  Source 4: USGS (Streamflow discharge)")
print("  Source 5: NOAA OISST v2.1 NetCDF (Spatial gradients)")


[SOURCE 5] NASA OISST - Spatial SST Gradient Metrics from NetCDF Grids
--------------------------------------------------------------------------------

  Accessing NOAA OISST v2.1 via NCEI THREDDS OPeNDAP server...
  Computing spatial metrics for Sep 1981 - Jan 2026 (approximately 533 months)
  This will take 10-15 minutes as we process gridded NetCDF data remotely...

  Processing spatial metrics month by month...
    Processed 50 months (50 successful)...
    Processed 100 months (100 successful)...
    Processed 150 months (150 successful)...
    Processed 200 months (200 successful)...
    Processed 250 months (250 successful)...
    Processed 300 months (300 successful)...
    Processed 350 months (350 successful)...
    Processed 400 months (400 successful)...
    Processed 450 months (450 successful)...
    Processed 500 months (500 successful)...

 SOURCE 5 COMPLETE: 533 monthly observations (1981-2026)
  Columns: ['year_month', 'zonal_gradient', 'meridional_asymmetry', 'cold

#TRANSFORM PHASE

###STEP 1: Create time_dimension master table

In [ ]:
print("\nTRANSFORM STEP 1: Creating time_dimension master table")
print("-" * 80)

# Collect all unique year_month values across all sources
all_year_months = set()

for df in [sst_observations, atmospheric_state, oscillation_indices,
           hydrological_impacts, spatial_metrics]:
    if 'year_month' in df.columns:
        all_year_months.update(df['year_month'].tolist())

# Create time_dimension DataFrame
time_dimension_data = []
for ym in sorted(all_year_months):
    year = int(ym.split('-')[0])
    month = int(ym.split('-')[1])

    # Determine season (meteorological seasons)
    if month in [12, 1, 2]:
        season = 'DJF'  # Winter
    elif month in [3, 4, 5]:
        season = 'MAM'  # Spring
    elif month in [6, 7, 8]:
        season = 'JJA'  # Summer
    else:  # [9, 10, 11]
        season = 'SON'  # Fall

    # Determine the decade
    decade = (year // 10) * 10

    time_dimension_data.append({
        'year_month': ym,
        'year': year,
        'month': month,
        'season': season,
        'decade': decade
    })

time_dimension = pd.DataFrame(time_dimension_data)

print(f"  Created time_dimension: {len(time_dimension)} unique months")
print(f"  Date range: {time_dimension['year_month'].min()} to {time_dimension['year_month'].max()}")
print(f"  Decades covered: {sorted(time_dimension['decade'].unique())}")
print("\n  Sample:")
print(time_dimension.head())


TRANSFORM STEP 1: Creating time_dimension master table
--------------------------------------------------------------------------------
  Created time_dimension: 902 unique months
  Date range: 1951-01 to 2026-02
  Decades covered: [np.int64(1950), np.int64(1960), np.int64(1970), np.int64(1980), np.int64(1990), np.int64(2000), np.int64(2010), np.int64(2020)]

  Sample:
  year_month  year  month season  decade
0    1951-01  1951      1    DJF    1950
1    1951-02  1951      2    DJF    1950
2    1951-03  1951      3    MAM    1950
3    1951-04  1951      4    MAM    1950
4    1951-05  1951      5    MAM    1950


###STEP 2: Baseline normalization

Source 1 (SST) contains ABSOLUTE SST temperatures (24-29°C), not anomalies. We intentionally keep raw SST values in sst_observations because:
1. End users (dynamical systems researchers) may want to define their own baseline periods
2. Anomaly computation happens in Transform Step 3 for ENSO event classification
3. This preserves maximum flexibility for nonlinear forecasting applications

For the other sources:
- Source 2 (Atmospheric): Already contains anomalies from NCEP/NCAR climatology
- Source 3 (Oscillation): Already standardized indices (SOI, PDO, MJO)
- Source 4 (Streamflow): Converted to z-scores for cross-river comparison
- Source 5 (Spatial): Absolute SST values (gradients computed from raw temperatures)

Design decision: We use OUTER JOINs when merging sources to preserve all available temporal coverage. This means some months have partial data (e.g., spatial_metrics only covers 1981+), but this is appropriate because:
1. Different reanalysis products have different start dates
2. End users can filter to their desired overlap period via SQL queries
3. Preserving maximum temporal span (1951-2026) enables decadal-scale analysis

In [ ]:
print("\nTRANSFORM STEP 2: Baseline normalization")
print("-" * 80)
print(" Data normalization strategy:")
print("  - Source 1 (SST): Raw absolute temperatures stored; anomalies computed in Step 3")
print("  - Source 2 (Atmospheric): NCEP/NCAR reanalysis anomalies (relative to model climatology)")
print("  - Source 3 (Oscillation): Standardized indices (dimensionless)")
print("  - Source 4 (Streamflow): Z-scores (enables cross-basin comparison)")
print("  - Source 5 (Spatial): Absolute SST values (gradients preserve temperature units)")
print("\n Join strategy: OUTER joins preserve maximum temporal coverage (1951-2026)")
print(" All sources appropriately normalized for their intended use!")


TRANSFORM STEP 2: Baseline normalization
--------------------------------------------------------------------------------
 Data normalization strategy:
  - Source 1 (SST): Raw absolute temperatures stored; anomalies computed in Step 3
  - Source 2 (Atmospheric): NCEP/NCAR reanalysis anomalies (relative to model climatology)
  - Source 3 (Oscillation): Standardized indices (dimensionless)
  - Source 4 (Streamflow): Z-scores (enables cross-basin comparison)
  - Source 5 (Spatial): Absolute SST values (gradients preserve temperature units)

 Join strategy: OUTER joins preserve maximum temporal coverage (1951-2026)
 All sources appropriately normalized for their intended use!


###STEP 3: Derive enso_events table

In [ ]:
print("\nTRANSFORM STEP 3: Deriving enso_events table")
print("-" * 80)

# ENSO events are classified using NOAA CPC operational definition:
# El Niño/La Niña = Niño 3.4 anomaly is greater than or equal to positive or negative 0.5 degrees Celsius for 5 consecutive 3-month periods

print("  Applying NOAA CPC ENSO classification criteria:")
print("  - Niño 3.4 anomaly is greater than or equal to 0.5°C for 5 consecutive overlapping 3-month periods = El Niño")
print("  - Niño 3.4 anomaly is less than or equal to -0.5°C for 5 consecutive overlapping 3-month periods = La Niña")

# Get Niño 3.4 data
nino34_series = sst_observations[['year_month', 'nino34']].copy()
nino34_series = nino34_series.sort_values('year_month').reset_index(drop=True)

# CRITICAL: The data from NOAA PSL contains absolute SST values, not anomalies
# We need to compute anomalies relative to climatological baseline (1981-2010)
print("\n  Computing Niño 3.4 anomalies relative to 1981-2010 baseline...")

# Extract years for baseline period
nino34_series['year'] = nino34_series['year_month'].str[:4].astype(int)
nino34_series['month'] = nino34_series['year_month'].str[5:7].astype(int)

# Compute climatology (mean for each calendar month during 1981-2010)
baseline_data = nino34_series[(nino34_series['year'] >= 1981) & (nino34_series['year'] <= 2010)]
climatology = baseline_data.groupby('month')['nino34'].mean()

# Compute anomalies by subtracting the climatological mean for each month
nino34_series['nino34_anomaly'] = nino34_series.apply(
    lambda row: row['nino34'] - climatology[row['month']],
    axis=1
)

print(f"  Climatology computed from {len(baseline_data)} months (1981-2010)")
print(f"  Mean SST by month: {climatology.round(2).to_dict()}")
print(f"  Anomaly range: {nino34_series['nino34_anomaly'].min():.2f}°C to {nino34_series['nino34_anomaly'].max():.2f}°C")

# Compute 3-month running mean of anomalies
nino34_series['nino34_3month'] = nino34_series['nino34_anomaly'].rolling(window=3, center=True).mean()

# Identify ENSO conditions (threshold = ±0.5°C)
nino34_series['condition'] = 'Neutral'
nino34_series.loc[nino34_series['nino34_3month'] >= 0.5, 'condition'] = 'El Niño'
nino34_series.loc[nino34_series['nino34_3month'] <= -0.5, 'condition'] = 'La Niña'

# Find consecutive sequences of 5+ months
events = []
current_event = None
consecutive_count = 0

for idx, row in nino34_series.iterrows():
    condition = row['condition']
    year_month = row['year_month']
    intensity = row['nino34_3month']

    if condition != 'Neutral':
        if current_event is None or current_event['event_type'] != condition:
            # Start new potential event
            current_event = {
                'event_type': condition,
                'start_month': year_month,
                'months': [year_month],
                'intensities': [intensity]
            }
            consecutive_count = 1
        else:
            # Continue current event
            current_event['months'].append(year_month)
            current_event['intensities'].append(intensity)
            consecutive_count += 1

            # If we have 5+ consecutive months, this is a valid ENSO event
            if consecutive_count >= 5 and 'end_month' not in current_event:
                current_event['end_month'] = year_month
                # Find peak intensity
                peak_idx = np.argmax(np.abs(current_event['intensities']))
                current_event['peak_month'] = current_event['months'][peak_idx]
                current_event['peak_intensity'] = current_event['intensities'][peak_idx]
    else:
        # Neutral conditions - finalize current event if it exists
        if current_event is not None and consecutive_count >= 5:
            # Event ended in previous month
            if 'end_month' not in current_event:
                current_event['end_month'] = current_event['months'][-1]
                peak_idx = np.argmax(np.abs(current_event['intensities']))
                current_event['peak_month'] = current_event['months'][peak_idx]
                current_event['peak_intensity'] = current_event['intensities'][peak_idx]

            events.append(current_event)

        current_event = None
        consecutive_count = 0

# Finalize last event if still active
if current_event is not None and consecutive_count >= 5:
    if 'end_month' not in current_event:
        current_event['end_month'] = current_event['months'][-1]
        peak_idx = np.argmax(np.abs(current_event['intensities']))
        current_event['peak_month'] = current_event['months'][peak_idx]
        current_event['peak_intensity'] = current_event['intensities'][peak_idx]
    events.append(current_event)

# Create enso_events DataFrame
enso_events_data = []
for i, event in enumerate(events, start=1):
    enso_events_data.append({
        'event_id': i,
        'start_month': event['start_month'],
        'end_month': event['end_month'],
        'peak_month': event['peak_month'],
        'event_type': event['event_type'],
        'peak_intensity': event['peak_intensity']
    })

enso_events = pd.DataFrame(enso_events_data)

print(f"\n  Identified {len(enso_events)} ENSO events (1951-2026)")
print(f"  - El Niño events: {len(enso_events[enso_events['event_type'] == 'El Niño'])}")
print(f"  - La Niña events: {len(enso_events[enso_events['event_type'] == 'La Niña'])}")
print("\n  Sample events:")
print(enso_events.head(10))


TRANSFORM STEP 3: Deriving enso_events table
--------------------------------------------------------------------------------
  Applying NOAA CPC ENSO classification criteria:
  - Niño 3.4 anomaly is greater than or equal to 0.5°C for 5 consecutive overlapping 3-month periods = El Niño
  - Niño 3.4 anomaly is less than or equal to -0.5°C for 5 consecutive overlapping 3-month periods = La Niña

  Computing Niño 3.4 anomalies relative to 1981-2010 baseline...
  Climatology computed from 360 months (1981-2010)
  Mean SST by month: {1: 26.55, 2: 26.74, 3: 27.23, 4: 27.71, 5: 27.81, 6: 27.59, 7: 27.18, 8: 26.83, 9: 26.73, 10: 26.67, 11: 26.63, 12: 26.56}
  Anomaly range: -2.38°C to 2.79°C

  Identified 37 ENSO events (1951-2026)
  - El Niño events: 16
  - La Niña events: 21

  Sample events:
   event_id start_month end_month peak_month event_type  peak_intensity
0         1     1954-04   1954-08    1954-08    La Niña       -1.360000
1         2     1957-05   1957-09    1957-07    El Niño  

In [ ]:
# Data quality and missingness summary
print("DATA QUALITY ASSESSMENT")
print("-" * 80)

# Quantify missing data across all sources
print("\nMissing data by source:")
print(f"  Source 1 (SST): {sst_observations.isnull().sum().sum()} / {sst_observations.size} values ({100*sst_observations.isnull().sum().sum()/sst_observations.size:.1f}%)")
print(f"  Source 2 (Atmospheric): {atmospheric_state.isnull().sum().sum()} / {atmospheric_state.size} values ({100*atmospheric_state.isnull().sum().sum()/atmospheric_state.size:.1f}%)")
print(f"  Source 3 (Oscillation): {oscillation_indices.isnull().sum().sum()} / {oscillation_indices.size} values ({100*oscillation_indices.isnull().sum().sum()/oscillation_indices.size:.1f}%)")
print(f"  Source 4 (Streamflow): {hydrological_impacts.isnull().sum().sum()} / {hydrological_impacts.size} values ({100*hydrological_impacts.isnull().sum().sum()/hydrological_impacts.size:.1f}%)")
print(f"  Source 5 (Spatial): {spatial_metrics.isnull().sum().sum()} / {spatial_metrics.size} values ({100*spatial_metrics.isnull().sum().sum()/spatial_metrics.size:.1f}%)")

# Critical limitation: atmospheric wind variables
u_coverage = atmospheric_state[['u850', 'u200']].notna().all(axis=1).sum()
print(f"\nKNOWN LIMITATION - Atmospheric wind coverage:")
print(f"  u850 and u200 available for {u_coverage} / {len(atmospheric_state)} months ({100*u_coverage/len(atmospheric_state):.1f}%)")
print(f"  Coverage period: approximately 1949-1982 (IRI Data Library limitation)")
print(f"  Impact: Walker Index missing for post-1982 El Niño/La Niña events")
print(f"  Justification: This reflects actual data availability in NCEP/NCAR Reanalysis")

# OLR coverage
olr_coverage = atmospheric_state['olr'].notna().sum()
print(f"\n  OLR available for {olr_coverage} / {len(atmospheric_state)} months ({100*olr_coverage/len(atmospheric_state):.1f}%)")
print(f"  Impact: Limited outgoing longwave radiation data")

# MJO temporal coverage
mjo_coverage = oscillation_indices[['mjo_rmm1', 'mjo_rmm2']].notna().all(axis=1).sum()
print(f"\nMJO coverage:")
print(f"  Available for {mjo_coverage} / {len(oscillation_indices)} months ({100*mjo_coverage/len(oscillation_indices):.1f}%)")
print(f"  Coverage period: 1974-2024 (MJO index development date)")

print("\n Data quality assessment is complete!")

DATA QUALITY ASSESSMENT
--------------------------------------------------------------------------------

Missing data by source:
  Source 1 (SST): 2 / 5412 values (0.0%)
  Source 2 (Atmospheric): 2421 / 6307 values (38.4%)
  Source 3 (Oscillation): 1222 / 6314 values (19.4%)
  Source 4 (Streamflow): 0 / 3608 values (0.0%)
  Source 5 (Spatial): 0 / 2665 values (0.0%)

KNOWN LIMITATION - Atmospheric wind coverage:
  u850 and u200 available for 381 / 901 months (42.3%)
  Coverage period: approximately 1949-1982 (IRI Data Library limitation)
  Impact: Walker Index missing for post-1982 El Niño/La Niña events
  Justification: This reflects actual data availability in NCEP/NCAR Reanalysis

  OLR available for 40 / 901 months (4.4%)
  Impact: Limited outgoing longwave radiation data

MJO coverage:
  Available for 597 / 902 months (66.2%)
  Coverage period: 1974-2024 (MJO index development date)

 Data quality assessment is complete!


###STEP 4: Final data quality checks

In [ ]:

print("\nTRANSFORM STEP 4: Data quality checks")
print("-" * 80)

print("\nTable summaries:")
print(f"  time_dimension: {len(time_dimension)} rows, {time_dimension.columns.tolist()}")
print(f"  sst_observations: {len(sst_observations)} rows, {sst_observations.columns.tolist()}")
print(f"  atmospheric_state: {len(atmospheric_state)} rows, {atmospheric_state.columns.tolist()}")
print(f"  oscillation_indices: {len(oscillation_indices)} rows, {oscillation_indices.columns.tolist()}")
print(f"  hydrological_impacts: {len(hydrological_impacts)} rows, {hydrological_impacts.columns.tolist()}")
print(f"  spatial_metrics: {len(spatial_metrics)} rows, {spatial_metrics.columns.tolist()}")
print(f"  enso_events: {len(enso_events)} rows, {enso_events.columns.tolist()}")

print("\n All tables have been validated and are ready for database loading")

print(" TRANSFORM PHASE IS NOW FULLY COMPLETE.")


TRANSFORM STEP 4: Data quality checks
--------------------------------------------------------------------------------

Table summaries:
  time_dimension: 902 rows, ['year_month', 'year', 'month', 'season', 'decade']
  sst_observations: 902 rows, ['year_month', 'nino12', 'nino3', 'nino34', 'nino4', 'tni']
  atmospheric_state: 901 rows, ['year_month', 'u850', 'u200', 'slp_darwin', 'slp_tahiti', 'olr', 'walker_index']
  oscillation_indices: 902 rows, ['year_month', 'soi', 'pdo', 'mjo_rmm1', 'mjo_rmm2', 'mjo_amplitude', 'mjo_phase']
  hydrological_impacts: 902 rows, ['year_month', 'colorado_discharge', 'columbia_discharge', 'sacramento_discharge']
  spatial_metrics: 533 rows, ['year_month', 'zonal_gradient', 'meridional_asymmetry', 'cold_tongue_intensity', 'quality_flag']
  enso_events: 37 rows, ['event_id', 'start_month', 'end_month', 'peak_month', 'event_type', 'peak_intensity']

 All tables have been validated and are ready for database loading
 TRANSFORM PHASE IS NOW FULLY COMPLETE.


#LOAD PHASE

###STEP 1: Drop existing tables (if there are any)

In [ ]:
print("\nLOAD STEP 1: Dropping existing tables")
print("-" * 80)

try:
    conn = get_db_connection()
    cursor = conn.cursor()

    # Drop tables in reverse dependency order
    tables_to_drop = [
        'enso_events',
        'spatial_metrics',
        'hydrological_impacts',
        'oscillation_indices',
        'atmospheric_state',
        'sst_observations',
        'time_dimension'
    ]

    for table in tables_to_drop:
        cursor.execute(f"DROP TABLE IF EXISTS {table} CASCADE;")
        print(f"  Dropped table: {table}")

    conn.commit()
    cursor.close()
    conn.close()

    print(" All existing tables dropped successfully")

except Exception as e:
    print(f" Error dropping tables: {e}")


LOAD STEP 1: Dropping existing tables
--------------------------------------------------------------------------------
  Dropped table: enso_events
  Dropped table: spatial_metrics
  Dropped table: hydrological_impacts
  Dropped table: oscillation_indices
  Dropped table: atmospheric_state
  Dropped table: sst_observations
  Dropped table: time_dimension
 All existing tables dropped successfully


###STEP 2: Create the database schema

In [ ]:
from IPython.display import Image, display
display(Image('/content/ENSO DB Schema.png', width=800))

FileNotFoundError: No such file or directory: '/content/ENSO DB Schema.png'

FileNotFoundError: No such file or directory: '/content/ENSO DB Schema.png'

<IPython.core.display.Image object>

In [ ]:
print("\nLOAD STEP 2: Creating database schema")
print("-" * 80)

# SQL statements to create all tables
create_table_sql = """
-- Master time dimension table
CREATE TABLE time_dimension (
    year_month VARCHAR(7) PRIMARY KEY,
    year INTEGER NOT NULL,
    month INTEGER NOT NULL,
    season VARCHAR(3) NOT NULL,
    decade INTEGER NOT NULL
);

-- SST observations (Source 1: NOAA PSL ERSSTv5)
CREATE TABLE sst_observations (
    year_month VARCHAR(7) PRIMARY KEY REFERENCES time_dimension(year_month),
    nino12 FLOAT,
    nino3 FLOAT,
    nino34 FLOAT,
    nino4 FLOAT,
    tni FLOAT
);

-- Atmospheric state (Source 2: NCEP/NCAR Reanalysis)
CREATE TABLE atmospheric_state (
    year_month VARCHAR(7) PRIMARY KEY REFERENCES time_dimension(year_month),
    u850 FLOAT,
    u200 FLOAT,
    slp_darwin FLOAT,
    slp_tahiti FLOAT,
    olr FLOAT,
    walker_index FLOAT
);

-- Oscillation indices (Source 3: NOAA CPC)
CREATE TABLE oscillation_indices (
    year_month VARCHAR(7) PRIMARY KEY REFERENCES time_dimension(year_month),
    soi FLOAT,
    pdo FLOAT,
    mjo_rmm1 FLOAT,
    mjo_rmm2 FLOAT,
    mjo_amplitude FLOAT,
    mjo_phase INTEGER
);

-- Hydrological impacts (Source 4: USGS)
CREATE TABLE hydrological_impacts (
    year_month VARCHAR(7) PRIMARY KEY REFERENCES time_dimension(year_month),
    colorado_discharge FLOAT,
    columbia_discharge FLOAT,
    sacramento_discharge FLOAT
);

-- Spatial metrics (Source 5: NOAA OISST NetCDF)
CREATE TABLE spatial_metrics (
    year_month VARCHAR(7) PRIMARY KEY REFERENCES time_dimension(year_month),
    zonal_gradient FLOAT,
    meridional_asymmetry FLOAT,
    cold_tongue_intensity FLOAT,
    quality_flag INTEGER
);

-- ENSO events (Derived from sst_observations)
CREATE TABLE enso_events (
    event_id INTEGER PRIMARY KEY,
    start_month VARCHAR(7) REFERENCES time_dimension(year_month),
    end_month VARCHAR(7) REFERENCES time_dimension(year_month),
    peak_month VARCHAR(7) REFERENCES time_dimension(year_month),
    event_type VARCHAR(10) NOT NULL,
    peak_intensity FLOAT NOT NULL
);
"""

try:
    conn = get_db_connection()
    cursor = conn.cursor()

    # Execute schema creation
    cursor.execute(create_table_sql)
    conn.commit()

    print(" All tables created successfully:")
    print("  - time_dimension (master table)")
    print("  - sst_observations")
    print("  - atmospheric_state")
    print("  - oscillation_indices")
    print("  - hydrological_impacts")
    print("  - spatial_metrics")
    print("  - enso_events")

    cursor.close()
    conn.close()

except Exception as e:
    print(f" Error creating schema: {e}")


LOAD STEP 2: Creating database schema
--------------------------------------------------------------------------------
 All tables created successfully:
  - time_dimension (master table)
  - sst_observations
  - atmospheric_state
  - oscillation_indices
  - hydrological_impacts
  - spatial_metrics
  - enso_events


###STEP 3: Load data into tables (this could also take a while)

In [ ]:
print("\nLOAD STEP 3: Loading data into PostgreSQL")
print("-" * 80)

# First we will load a pandas DataFrame into a PostgreSQL table
# This function uses bulk insert for efficiency purposes
def load_dataframe_to_postgres(df, table_name, conn):
    cursor = conn.cursor()

    # Convert DataFrame to list of tuples
    columns = df.columns.tolist()
    values = df.values.tolist()

    # Build INSERT statement
    placeholders = ', '.join(['%s'] * len(columns))
    columns_str = ', '.join(columns)
    insert_sql = f"INSERT INTO {table_name} ({columns_str}) VALUES ({placeholders})"

    # Execute bulk insert
    cursor.executemany(insert_sql, values)
    conn.commit()
    cursor.close()

try:
    conn = get_db_connection()

    # Load tables in dependency order
    print("  Loading time_dimension...")
    load_dataframe_to_postgres(time_dimension, 'time_dimension', conn)
    print(f"    Successfully loaded {len(time_dimension)} rows")

    print("  Loading sst_observations...")
    load_dataframe_to_postgres(sst_observations, 'sst_observations', conn)
    print(f"    Successfully loaded {len(sst_observations)} rows")

    print("  Loading atmospheric_state...")
    load_dataframe_to_postgres(atmospheric_state, 'atmospheric_state', conn)
    print(f"    Successfully loaded {len(atmospheric_state)} rows")

    print("  Loading oscillation_indices...")
    # Note that for mjo_phase, I had to convert NaN to NULL for PostgreSQL INTEGER compatibility
    oscillation_indices_clean = oscillation_indices.copy()
    oscillation_indices_clean['mjo_phase'] = oscillation_indices_clean['mjo_phase'].replace({np.nan: None})
    load_dataframe_to_postgres(oscillation_indices_clean, 'oscillation_indices', conn)
    print(f"    Successfully loaded {len(oscillation_indices)} rows")

    print("  Loading hydrological_impacts...")
    load_dataframe_to_postgres(hydrological_impacts, 'hydrological_impacts', conn)
    print(f"    Successfully loaded {len(hydrological_impacts)} rows")

    print("  Loading spatial_metrics...")
    load_dataframe_to_postgres(spatial_metrics, 'spatial_metrics', conn)
    print(f"    Successfully loaded {len(spatial_metrics)} rows")

    print("  Loading enso_events...")
    load_dataframe_to_postgres(enso_events, 'enso_events', conn)
    print(f"    Successfully loaded {len(enso_events)} rows")

    conn.close()
    print("\n All data loaded successfully!")

except Exception as e:
    print(f" Error loading data: {e}")


LOAD STEP 3: Loading data into PostgreSQL
--------------------------------------------------------------------------------
  Loading time_dimension...
    Successfully loaded 902 rows
  Loading sst_observations...
    Successfully loaded 902 rows
  Loading atmospheric_state...
    Successfully loaded 901 rows
  Loading oscillation_indices...
    Successfully loaded 902 rows
  Loading hydrological_impacts...
    Successfully loaded 902 rows
  Loading spatial_metrics...
    Successfully loaded 533 rows
  Loading enso_events...
    Successfully loaded 37 rows

 All data loaded successfully!


###STEP 4: Verify database integrity

In [ ]:
print("\nLOAD STEP 4: Verifying database integrity")
print("-" * 80)

try:
    conn = get_db_connection()
    cursor = conn.cursor()

    # Check row counts
    tables = ['time_dimension', 'sst_observations', 'atmospheric_state',
              'oscillation_indices', 'hydrological_impacts', 'spatial_metrics', 'enso_events']

    print("\nTable row counts:")
    for table in tables:
        cursor.execute(f"SELECT COUNT(*) FROM {table};")
        count = cursor.fetchone()[0]
        print(f"  {table}: {count} rows")

    # Test a sample query
    print("\nSample query: El Niño events with peak intensity > 1.5°C")
    cursor.execute("""
        SELECT event_id, start_month, end_month, peak_intensity
        FROM enso_events
        WHERE event_type = 'El Niño' AND peak_intensity > 1.5
        ORDER BY peak_intensity DESC
        LIMIT 5;
    """)
    results = cursor.fetchall()
    print("  Results:")
    for row in results:
        print(f"    Event {row[0]}: {row[1]} to {row[2]}, peak={row[3]:.2f}°C")

    # Test a join query
    print("\nSample join query: Strong El Niño months with atmospheric conditions")
    cursor.execute("""
        SELECT e.event_id, e.peak_month, s.nino34, a.walker_index
        FROM enso_events e
        JOIN sst_observations s ON e.peak_month = s.year_month
        JOIN atmospheric_state a ON e.peak_month = a.year_month
        WHERE e.event_type = 'El Niño' AND e.peak_intensity > 1.5
        ORDER BY e.peak_intensity DESC
        LIMIT 5;
    """)
    results = cursor.fetchall()
    print("  Results:")
    for row in results:
        print(f"    Event {row[0]} ({row[1]}): Niño 3.4={row[2]:.2f}°C, Walker Index={row[3]:.2f}")

    cursor.close()
    conn.close()

    print("\n Database integrity has been successfully verified!")

except Exception as e:
    print(f" Error verifying database: {e}")

print("LOAD PHASE COMPLETE. The database is now fully operational!")
print("="*80)
print("\nDatabase details:")
print(f"  Host: {DB_CONFIG['host']}")
print(f"  Database: {DB_CONFIG['database']}")
print(f"  Tables: 7 (1 master + 5 observation + 1 derived)")
print(f"  Total rows: ~{len(time_dimension) + len(sst_observations) + len(atmospheric_state) + len(oscillation_indices) + len(hydrological_impacts) + len(spatial_metrics) + len(enso_events)}")
print(f"  Temporal coverage: {time_dimension['year_month'].min()} to {time_dimension['year_month'].max()}")


LOAD STEP 4: Verifying database integrity
--------------------------------------------------------------------------------

Table row counts:
  time_dimension: 902 rows
  sst_observations: 902 rows
  atmospheric_state: 901 rows
  oscillation_indices: 902 rows
  hydrological_impacts: 902 rows
  spatial_metrics: 533 rows
  enso_events: 37 rows

Sample query: El Niño events with peak intensity > 1.5°C
  Results:
    Event 22: 1997-05 to 1997-09, peak=2.15°C
    Event 32: 2015-03 to 2015-07, peak=1.61°C
    Event 11: 1972-06 to 1972-10, peak=1.60°C
    Event 37: 2023-05 to 2023-09, peak=1.58°C

Sample join query: Strong El Niño months with atmospheric conditions
  Results:
    Event 22 (1997-09): Niño 3.4=28.85°C, Walker Index=nan
    Event 32 (2015-07): Niño 3.4=28.75°C, Walker Index=nan
    Event 11 (1972-10): Niño 3.4=28.26°C, Walker Index=-0.43
    Event 37 (2023-09): Niño 3.4=28.32°C, Walker Index=nan

 Database integrity has been successfully verified!
LOAD PHASE COMPLETE. The datab

##**PROJECT COMPLETE**

The ENSO relational database is now fully operational and ready for nonlinear dynamical systems analysis.

### **What Was Built**

This ETL pipeline successfully integrated 5 data sources from 4 different agencies:
- NOAA PSL - Sea surface temperatures (ERSSTv5)
- NCEP/NCAR - Atmospheric reanalysis (wind, pressure, radiation)
- NOAA CPC - Climate oscillation indices (SOI, PDO, MJO)
- USGS - Streamflow discharge from 3 ENSO-sensitive river basins
- NASA OISST - Gridded SST spatial gradients (via OPeNDAP/NetCDF processing)

### **Database Specifications**

- Tables: 7 (1 master time dimension + 5 observation tables + 1 derived events table)
- Total Records: 5,077 rows across all tables
- Temporal Coverage: 1951-2026 (75 years)
- Schema Design: Star schema with time_dimension as the hub
- ENSO Events Classified: 37 events (16 El Niño, 21 La Niña)


### **Database Is Now Ready For...**

- Nonlinear time series forecasting (reservoir computing, LSTM, transformers)
- ENSO teleconnection analysis (hydrological impacts, atmospheric coupling)
- Phase space reconstruction and attractor analysis
- Multi-scale temporal pattern detection

In [ ]:
# DATABASE VISUALIZATION (for fun)
print("DATABASE SAMPLE VISUALIZATION")


import pandas as pd

# Connect to database
conn = get_db_connection()

# 1. Show complete schema structure
print("\n[1] COMPLETE SCHEMA STRUCTURE")
print("-" * 80)

schema_info = []
for table in ['time_dimension', 'sst_observations', 'atmospheric_state',
              'oscillation_indices', 'hydrological_impacts', 'spatial_metrics', 'enso_events']:
    cursor = conn.cursor()
    cursor.execute(f"""
        SELECT column_name, data_type
        FROM information_schema.columns
        WHERE table_name = '{table}'
        ORDER BY ordinal_position;
    """)
    columns = cursor.fetchall()
    cursor.close()

    print(f"\n{table.upper()}:")
    for col_name, col_type in columns:
        print(f"  - {col_name}: {col_type}")

# 2. Show sample joined data (all tables combined)
print("\n\n[2] SAMPLE JOINED DATA (All tables combined)")
print("-" * 80)

query = """
SELECT
    t.year_month,
    t.season,
    s.nino34,
    a.walker_index,
    o.soi,
    h.colorado_discharge,
    sm.zonal_gradient,
    e.event_type
FROM time_dimension t
LEFT JOIN sst_observations s ON t.year_month = s.year_month
LEFT JOIN atmospheric_state a ON t.year_month = a.year_month
LEFT JOIN oscillation_indices o ON t.year_month = o.year_month
LEFT JOIN hydrological_impacts h ON t.year_month = h.year_month
LEFT JOIN spatial_metrics sm ON t.year_month = sm.year_month
LEFT JOIN enso_events e ON t.year_month = e.peak_month
WHERE t.year_month >= '2020-01'
ORDER BY t.year_month
LIMIT 10;
"""

df_sample = pd.read_sql(query, conn)
print("\nRecent data (2020-present):")
print(df_sample.to_string(index=False))

# 3. Show ENSO event timeline
print("\n\n[3] ENSO EVENT TIMELINE (Recent events)")
print("-" * 80)

query_events = """
SELECT
    event_id,
    event_type,
    start_month,
    end_month,
    peak_month,
    ROUND(peak_intensity::numeric, 2) as peak_intensity
FROM enso_events
WHERE start_month >= '2010-01'
ORDER BY start_month;
"""

df_events = pd.read_sql(query_events, conn)
print(df_events.to_string(index=False))

# 4. Show data completeness by decade
print("\n\n[4] DATA COMPLETENESS BY DECADE")
print("-" * 80)

query_completeness = """
SELECT
    t.decade,
    COUNT(DISTINCT t.year_month) as total_months,
    COUNT(s.year_month) as sst_coverage,
    COUNT(a.u850) as wind_coverage,
    COUNT(sm.year_month) as spatial_coverage
FROM time_dimension t
LEFT JOIN sst_observations s ON t.year_month = s.year_month
LEFT JOIN atmospheric_state a ON t.year_month = a.year_month
LEFT JOIN spatial_metrics sm ON t.year_month = sm.year_month
GROUP BY t.decade
ORDER BY t.decade;
"""

df_completeness = pd.read_sql(query_completeness, conn)
print(df_completeness.to_string(index=False))

conn.close()

print("\nDatabase visualization complete!")

DATABASE SAMPLE VISUALIZATION

[1] COMPLETE SCHEMA STRUCTURE
--------------------------------------------------------------------------------

TIME_DIMENSION:
  - year_month: character varying
  - year: integer
  - month: integer
  - season: character varying
  - decade: integer

SST_OBSERVATIONS:
  - year_month: character varying
  - nino12: double precision
  - nino3: double precision
  - nino34: double precision
  - nino4: double precision
  - tni: double precision

ATMOSPHERIC_STATE:
  - year_month: character varying
  - u850: double precision
  - u200: double precision
  - slp_darwin: double precision
  - slp_tahiti: double precision
  - olr: double precision
  - walker_index: double precision

OSCILLATION_INDICES:
  - year_month: character varying
  - soi: double precision
  - pdo: double precision
  - mjo_rmm1: double precision
  - mjo_rmm2: double precision
  - mjo_amplitude: double precision
  - mjo_phase: integer

HYDROLOGICAL_IMPACTS:
  - year_month: character varying
  - co

###Final Data Queries

In [ ]:
print("LIVE QUERY DEMONSTRATIONS FOR END USERS")


conn = get_db_connection()

# QUERY 1: El Niño Impact Analysis - Streamflow Response
# Purpose: Identify which river basins show strongest discharge anomalies during El Niño events
print("\n[QUERY 1] El Niño Impact Analysis: Which river basins respond most to El Niño events?")
print("-" * 80)

query1 = """
SELECT
    e.event_id,
    e.event_type,
    e.start_month,
    e.end_month,
    ROUND(AVG(h.colorado_discharge)::numeric, 2) as avg_colorado_anomaly,
    ROUND(AVG(h.columbia_discharge)::numeric, 2) as avg_columbia_anomaly,
    ROUND(AVG(h.sacramento_discharge)::numeric, 2) as avg_sacramento_anomaly
FROM enso_events e
JOIN hydrological_impacts h ON h.year_month BETWEEN e.start_month AND e.end_month
WHERE e.event_type = 'El Niño'
GROUP BY e.event_id, e.event_type, e.start_month, e.end_month
ORDER BY e.start_month DESC
LIMIT 5;
"""

result1 = pd.read_sql(query1, conn)
print("\nResults (Recent El Niño Events):")
print(result1.to_string(index=False))
print("\nInterpretation: Positive z-scores = higher than normal discharge, negative = lower than normal")
print("Expected: Sacramento typically shows positive anomalies during El Niño (increased CA rainfall)")

# QUERY 2: Seasonal ENSO Pattern Analysis
# Purpose: Determine which seasons have the strongest ENSO signal
print("\n\n[QUERY 2] Seasonal ENSO Patterns: When are El Niño/La Niña events strongest?")
print("-" * 80)

query2 = """
SELECT
    t.season,
    e.event_type,
    COUNT(*) as event_count,
    ROUND(AVG(s.nino34)::numeric, 2) as avg_nino34_sst,
    ROUND(AVG(sm.zonal_gradient)::numeric, 2) as avg_zonal_gradient
FROM time_dimension t
JOIN enso_events e ON t.year_month = e.peak_month
JOIN sst_observations s ON t.year_month = s.year_month
LEFT JOIN spatial_metrics sm ON t.year_month = sm.year_month
GROUP BY t.season, e.event_type
ORDER BY e.event_type, event_count DESC;
"""

result2 = pd.read_sql(query2, conn)
print("\nResults (Seasonal Distribution):")
print(result2.to_string(index=False))
print("\nInterpretation: DJF (Dec-Jan-Feb) typically shows peak El Niño intensity")
print("Expected: Most ENSO events peak in boreal winter (DJF)")

# QUERY 3: Atmospheric-Ocean Coupling During ENSO
# Purpose: Examine Walker Circulation response during strong El Niño events
print("\n\n[QUERY 3] Atmospheric-Ocean Coupling: How does Walker Circulation respond to ENSO?")
print("-" * 80)

query3 = """
SELECT
    e.event_id,
    e.event_type,
    e.peak_month,
    ROUND(s.nino34::numeric, 2) as peak_sst,
    ROUND(a.walker_index::numeric, 2) as walker_index,
    o.soi,
    sm.zonal_gradient
FROM enso_events e
JOIN sst_observations s ON e.peak_month = s.year_month
JOIN atmospheric_state a ON e.peak_month = a.year_month
JOIN oscillation_indices o ON e.peak_month = o.year_month
LEFT JOIN spatial_metrics sm ON e.peak_month = sm.year_month
WHERE e.peak_intensity > 1.5 OR e.peak_intensity < -1.5
ORDER BY ABS(e.peak_intensity) DESC
LIMIT 8;
"""

result3 = pd.read_sql(query3, conn)
print("\nResults (Strong ENSO Events):")
print(result3.to_string(index=False))
print("\nInterpretation: Walker Index measures zonal wind strength (u200 - u850)")
print("Expected: Weakened Walker Circulation (lower index) during El Niño")
print("Note: Some NaN values due to atmospheric data coverage limitations (1949-1982)")

conn.close()

LIVE QUERY DEMONSTRATIONS FOR END USERS

[QUERY 1] El Niño Impact Analysis: Which river basins respond most to El Niño events?
--------------------------------------------------------------------------------

Results (Recent El Niño Events):
 event_id event_type start_month end_month  avg_colorado_anomaly  avg_columbia_anomaly  avg_sacramento_anomaly
       37    El Niño     2023-05   2023-09                  0.27                 -0.15                   -0.13
       34    El Niño     2018-09   2019-01                 -0.17                 -0.76                   -0.49
       32    El Niño     2015-03   2015-07                 -0.07                 -0.12                   -0.66
       29    El Niño     2009-07   2009-11                 -0.16                 -0.73                   -0.40
       26    El Niño     2006-09   2007-01                 -0.22                 -0.61                   -0.49

Interpretation: Positive z-scores = higher than normal discharge, negative = lower than nor

In [ ]:
conn = get_db_connection()
query = """
SELECT
    t.year_month,
    t.year,
    t.month,
    t.season,
    s.nino34
FROM time_dimension t
JOIN sst_observations s ON t.year_month = s.year_month
ORDER BY t.year_month;
"""
df1 = pd.read_sql(query, conn)
print(df1.to_json(orient='records'))

[{"year_month":"1951-01","year":1951,"month":1,"season":"DJF","nino34":25.24},{"year_month":"1951-02","year":1951,"month":2,"season":"DJF","nino34":25.71},{"year_month":"1951-03","year":1951,"month":3,"season":"MAM","nino34":26.9},{"year_month":"1951-04","year":1951,"month":4,"season":"MAM","nino34":27.58},{"year_month":"1951-05","year":1951,"month":5,"season":"MAM","nino34":27.92},{"year_month":"1951-06","year":1951,"month":6,"season":"JJA","nino34":27.73},{"year_month":"1951-07","year":1951,"month":7,"season":"JJA","nino34":27.6},{"year_month":"1951-08","year":1951,"month":8,"season":"JJA","nino34":27.02},{"year_month":"1951-09","year":1951,"month":9,"season":"SON","nino34":27.23},{"year_month":"1951-10","year":1951,"month":10,"season":"SON","nino34":27.2},{"year_month":"1951-11","year":1951,"month":11,"season":"SON","nino34":27.25},{"year_month":"1951-12","year":1951,"month":12,"season":"DJF","nino34":26.91},{"year_month":"1952-01","year":1952,"month":1,"season":"DJF","nino34":26.67

In [ ]:
query = """
SELECT event_id, event_type, start_month, end_month, peak_month,
       ROUND(peak_intensity::numeric, 3) as peak_intensity
FROM enso_events
ORDER BY start_month;
"""
df2 = pd.read_sql(query, conn)
print(df2.to_json(orient='records'))

[{"event_id":1,"event_type":"La Ni\u00f1a","start_month":"1954-04","end_month":"1954-08","peak_month":"1954-08","peak_intensity":-1.36},{"event_id":2,"event_type":"El Ni\u00f1o","start_month":"1957-05","end_month":"1957-09","peak_month":"1957-07","peak_intensity":0.872},{"event_id":3,"event_type":"La Ni\u00f1a","start_month":"1961-08","end_month":"1961-12","peak_month":"1961-09","peak_intensity":-0.851},{"event_id":4,"event_type":"La Ni\u00f1a","start_month":"1962-08","end_month":"1962-12","peak_month":"1962-12","peak_intensity":-0.85},{"event_id":5,"event_type":"El Ni\u00f1o","start_month":"1963-08","end_month":"1963-12","peak_month":"1963-12","peak_intensity":0.896},{"event_id":6,"event_type":"La Ni\u00f1a","start_month":"1964-04","end_month":"1964-08","peak_month":"1964-08","peak_intensity":-1.163},{"event_id":7,"event_type":"El Ni\u00f1o","start_month":"1965-07","end_month":"1965-11","peak_month":"1965-11","peak_intensity":1.463},{"event_id":8,"event_type":"La Ni\u00f1a","start_mon

In [ ]:
query = """
SELECT
    e.event_id,
    e.event_type,
    e.start_month,
    e.end_month,
    ROUND(AVG(h.colorado_discharge)::numeric, 4) as colorado,
    ROUND(AVG(h.columbia_discharge)::numeric, 4) as columbia,
    ROUND(AVG(h.sacramento_discharge)::numeric, 4) as sacramento
FROM enso_events e
JOIN hydrological_impacts h
    ON h.year_month BETWEEN e.start_month AND e.end_month
GROUP BY e.event_id, e.event_type, e.start_month, e.end_month
ORDER BY e.start_month;
"""
df3 = pd.read_sql(query, conn)
print(df3.to_json(orient='records'))
conn.close()

[{"event_id":1,"event_type":"La Ni\u00f1a","start_month":"1954-04","end_month":"1954-08","colorado":-0.1684,"columbia":1.7079,"sacramento":-0.121},{"event_id":2,"event_type":"El Ni\u00f1o","start_month":"1957-05","end_month":"1957-09","colorado":4.0811,"columbia":1.1472,"sacramento":-0.2704},{"event_id":3,"event_type":"La Ni\u00f1a","start_month":"1961-08","end_month":"1961-12","colorado":-0.5114,"columbia":-0.9223,"sacramento":-0.5082},{"event_id":4,"event_type":"La Ni\u00f1a","start_month":"1962-08","end_month":"1962-12","colorado":-0.7426,"columbia":-0.6442,"sacramento":-0.3077},{"event_id":5,"event_type":"El Ni\u00f1o","start_month":"1963-08","end_month":"1963-12","colorado":-1.4228,"columbia":-0.8634,"sacramento":-0.2104},{"event_id":6,"event_type":"La Ni\u00f1a","start_month":"1964-04","end_month":"1964-08","colorado":-1.0064,"columbia":1.3674,"sacramento":-0.2328},{"event_id":7,"event_type":"El Ni\u00f1o","start_month":"1965-07","end_month":"1965-11","colorado":-0.1634,"columbia